In [1]:
import cv2
import os
import numpy as np

input_root = "segmented_frames"
output_root = "tracked_segmented_frames"

os.makedirs(output_root, exist_ok=True)

# store previous object positions
object_centroids = {}
object_id = 0

def get_centroid(x, y, w, h):
    return (int(x + w/2), int(y + h/2))

for match in sorted(os.listdir(input_root)):

    match_input = os.path.join(input_root, match)
    match_output = os.path.join(output_root, match)

    if not os.path.isdir(match_input):
        continue

    os.makedirs(match_output, exist_ok=True)

    print("Processing:", match)

    object_centroids = {}  # reset per match
    object_id = 0

    for file in sorted(os.listdir(match_input)):

        path = os.path.join(match_input, file)
        frame = cv2.imread(path)

        if frame is None:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # threshold (since segmented is binary)
        _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)

        # find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        new_centroids = {}

        for c in contours:

            if cv2.contourArea(c) < 300:  # filter noise
                continue

            x, y, w, h = cv2.boundingRect(c)
            cx, cy = get_centroid(x, y, w, h)

            # MATCH WITH EXISTING IDS
            matched_id = None
            min_dist = 9999

            for obj_id, (px, py) in object_centroids.items():

                dist = np.sqrt((cx - px)**2 + (cy - py)**2)

                if dist < 50 and dist < min_dist:
                    min_dist = dist
                    matched_id = obj_id

            # if no match → new ID
            if matched_id is None:
                matched_id = object_id
                object_id += 1

            new_centroids[matched_id] = (cx, cy)

            # DRAW BOX
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(frame,
                        f"Player {matched_id}",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2)

        object_centroids = new_centroids

        save_path = os.path.join(match_output, file)
        cv2.imwrite(save_path, frame)

print("Tracking completed!")

Processing: .ipynb_checkpoints
Processing: match1
Processing: match2
Processing: match3
Tracking completed!
